In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer, OrdinalEncoder, RobustScaler
from sklearn.metrics import r2_score



In [5]:
sns.set_theme(style="whitegrid")
df = pd.read_excel('Tension Test_VIT.xlsx')
df = df.dropna(subset=['Second Stress', 'Second Strain'])
df['Fiber Type'] = df['Fiber Type'].fillna('Control')

encoder = OrdinalEncoder()
df['Fiber Type_Encoded'] = encoder.fit_transform(df[['Fiber Type']])
print(f"Dataset Loaded: {df.shape[0]} samples.")

Dataset Loaded: 506 samples.


In [14]:
zero_var = df[all_features].columns[df[all_features].nunique() <= 1].tolist()
print(f"Zero Variance (Junk): {zero_var}")

Zero Variance (Junk): ['Shape Factor']


In [15]:
correlations = df[all_features + ['Second Strain']].corr()['Second Strain'].sort_values()
print(correlations)

Sand                 -0.360374
W/B                  -0.197234
Diameter (mm)        -0.192807
C/B                  -0.163208
Fiber                -0.149988
Fly ash C            -0.090163
Water                -0.037033
Coarse Aggr.         -0.022095
GGBS                  0.000053
W/C                   0.012331
Fiber Type_Encoded    0.027080
Cement                0.051192
Fiber Volume          0.092534
Length (mm)           0.119928
Water Reducer / SP    0.152108
Fly ash F             0.209522
Silica Fume           0.339987
L/D                   0.340316
RI                    0.347872
Second Strain         1.000000
Shape Factor               NaN
Name: Second Strain, dtype: float64


In [20]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, RobustScaler, PowerTransformer, OrdinalEncoder

all_features = [
    'Fiber Type_Encoded', 'Fiber Volume', 'Length (mm)', 'Diameter (mm)', 'L/D',
    'Shape Factor', 'RI', 'Cement', 'Water', 'Sand', 'Fly ash C',
    'Fly ash F', 'GGBS', 'Coarse Aggr.', 'Silica Fume',
    'Water Reducer / SP', 'Fiber', 'C/B', 'W/C', 'W/B'
]

strain_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('reg', TransformedTargetRegressor(
        regressor=ExtraTreesRegressor(n_estimators=500, random_state=42, max_features='sqrt', bootstrap = True),
        transformer=QuantileTransformer(n_quantiles=300, output_distribution='normal', random_state= 42)
    ))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(strain_model, df[all_features], df['Second Strain'], cv=kf, scoring='r2')
print(f"Forward Model (Second Strain) R2: {scores.mean():.4f}")

Forward Model (Second Strain) R2: 0.5775
